In [ ]:
# GCP pose estimation - Kaggle runner (Accelerator: GPU T4/P100, Internet: ON for timm weights)
# Attach two Kaggle datasets: the code (this repo) and the data (train parts + test_dataset).
import glob, os, shutil
CODE = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/gcp/model.py", recursive=True)[0]))
os.chdir("/kaggle/working")
shutil.rmtree("cache", ignore_errors=True)   # stale cache from earlier runs
for f in ["gcp", "train.py", "infer.py", "eda.py"]:   # always refresh code from the dataset
    shutil.rmtree(f, ignore_errors=True) if os.path.isdir(f) else None
    (shutil.copytree if os.path.isdir(f"{CODE}/{f}") else shutil.copy)(f"{CODE}/{f}", f)
TRAIN = "/kaggle/input"   # searched recursively; label keys decide which images are used
TEST = glob.glob("/kaggle/input/**/test_dataset", recursive=True)[0]
print(CODE, TEST)

In [ ]:
!pip -q install timm

In [ ]:
# Train on all projects except one held-out project per class (honest unseen-site metrics), then predict
!python train.py --stage 1 --train-roots {TRAIN} --out s1.pt --epochs 20
!python train.py --stage 2 --train-roots {TRAIN} --out s2.pt --epochs 25
!python infer.py --eval --train-roots {TRAIN} --s1 s1.pt --s2 s2.pt
!python infer.py --images "{TEST}" --s1 s1.pt --s2 s2.pt --train-roots {TRAIN} --out predictions.json
shutil.rmtree("cache", ignore_errors=True)